In [108]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [109]:
# =========================================================
# ENSANUT 2023 - SOLO: CSVs + CATÁLOGOS + MAP (código→etiqueta)
# - Lee todos los CSV por componente
# - Lee catálogos (Variables/Valores) aunque no empiecen en A1
# - Construye maps: { variable -> { codigo: etiqueta } }
# - NO aplica mapeos ni construye matrices
# =========================================================

import re
import pandas as pd
from pathlib import Path
from openpyxl import load_workbook

# ---------- CONFIG ----------
BASE_DIR = Path("/content/drive/MyDrive/POSTDOCTORADO C3/ENSANUT/2023/ensanut_2023_ALL")
COMPONENTES = ["nutricion", "salud"]  # ajusta si quieres
CSV_SUBDIR = "csv"
CAT_SUBDIR = "catalogos"

# ---------- Utilidad: detectar fila de encabezados ----------
def find_header_row(ws, expected_headers, search_rows=200, search_cols=30, min_hits=2):
    """
    Escanea las primeras 'search_rows' x 'search_cols' celdas
    y retorna el número de fila (1-indexado) donde aparezcan al menos 'min_hits'
    encabezados esperados (texto coincidente exacto).
    """
    for r, row in enumerate(ws.iter_rows(min_row=1, max_row=search_rows, max_col=search_cols), start=1):
        vals = [str(c.value).strip() if c.value is not None else "" for c in row]
        hits = sum(1 for h in expected_headers if h in vals)
        if hits >= min_hits:
            return r
    return None

# ---------- Lectura robusta de catálogo ----------
def read_catalogo(path_xlsx):
    """
    Devuelve:
      - df_variables  (hoja 'Variables' o la primera)
      - df_valores    (hoja 'Valores' si existe, ya normalizada)
      - maps          dict { variable -> {codigo: etiqueta} }
      - has_valores   bool
    """
    wb = load_workbook(path_xlsx, read_only=True, data_only=True)
    names = wb.sheetnames

    # ---- Variables ----
    sheet_vars = "Variables" if "Variables" in names else names[0]
    ws_vars = wb[sheet_vars]
    hdr_vars = find_header_row(
        ws_vars,
        expected_headers=["Variable", "Posición", "Posicion", "Etiqueta", "Nivel", "Ancho"],
        min_hits=2
    )
    if hdr_vars is None:
        raise ValueError(f"No se halló encabezado en hoja '{sheet_vars}' de {path_xlsx}")

    df_variables = pd.read_excel(
        path_xlsx, sheet_name=sheet_vars, engine="openpyxl",
        header=0, skiprows=hdr_vars-1
    )

    # Normalizar nombres de columnas
    renv = {}
    for c in df_variables.columns:
        cl = str(c).strip().lower()
        if cl == "variable":                renv[c] = "variable"
        elif cl in ("posición","posicion"): renv[c] = "posicion"
        elif "tiqueta" in cl:               renv[c] = "etiqueta"
        elif "nivel" in cl:                 renv[c] = "nivel"
        elif "ancho" in cl or "width" in cl:renv[c] = "ancho"
    if renv:
        df_variables = df_variables.rename(columns=renv)

    if "variable" in df_variables.columns:
        df_variables["variable"] = df_variables["variable"].astype(str).str.strip()

    # ---- Valores (opcional) ----
    has_valores = "Valores" in names
    df_valores = pd.DataFrame(columns=["variable","codigo","etiqueta"])
    maps = {}

    if has_valores:
        ws_vals = wb["Valores"]
        hdr_vals = find_header_row(
            ws_vals,
            expected_headers=["Variable","Valor","Código","Codigo","Etiqueta","Descripción","Descripcion","Label"],
            min_hits=2
        )
        if hdr_vals is not None:
            raw_vals = pd.read_excel(
                path_xlsx, sheet_name="Valores", engine="openpyxl",
                header=0, skiprows=hdr_vals-1
            )

            # Normalizar columnas típicas a: variable / codigo / etiqueta
            ren = {}
            for c in raw_vals.columns:
                cl = str(c).strip().lower()
                if cl in ("variable", "valor"):                            ren[c] = "variable"
                elif cl in ("código","codigo","code","valor","unnamed: 1"): ren[c] = "codigo"
                elif ("tiqueta" in cl) or ("label" in cl) or ("descrip" in cl):
                    ren[c] = "etiqueta"

            keep = [k for k in ["variable","codigo","etiqueta"] if k in set(ren.values())]
            if keep:
                df_valores = raw_vals.rename(columns=ren)[keep].copy()

                # Limpiar types
                if "variable" in df_valores:
                    df_valores["variable"] = df_valores["variable"].ffill().astype(str).str.strip()
                if "codigo" in df_valores:
                    # Mantener como string para preservar ceros a la izquierda y consistencia
                    df_valores["codigo"] = df_valores["codigo"].astype(str).str.strip()
                if "etiqueta" in df_valores:
                    df_valores["etiqueta"] = df_valores["etiqueta"].astype(str).str.strip()

                # Construir maps: última ocurrencia gana (evita duplicados desordenados)
                for var, sub in df_valores.groupby("variable", dropna=False):
                    kv = {}
                    for _, r in sub.iterrows():
                        cod = r.get("codigo")
                        lab = r.get("etiqueta")
                        if cod is not None and lab is not None:
                            kv[cod] = lab
                    if kv:
                        maps[var] = kv

    return df_variables, df_valores, maps, has_valores

# ---------- Carga por componente: CSV + CATÁLOGOS + MAP ----------
def cargar_componentes_min(BASE_DIR: Path, componentes, csv_subdir="csv", cat_subdir="catalogos"):
    """
    Retorna un dict:
      dfs = {
        componente: {
          "csv": { nombre_tabla: DataFrame, ... },
          "catalogos": { nombre_catalogo: {"variables": df_vars, "valores": df_vals}, ... },
          "maps": { variable: {codigo: etiqueta}, ... }
        }, ...
      }
    """
    dfs = {c: {"csv": {}, "catalogos": {}, "maps": {}} for c in componentes}

    for comp in componentes:
        comp_dir = BASE_DIR / comp
        print(f"\n===== COMPONENTE: {comp.upper()} =====")
        print(f"Base: {comp_dir}")

        # --- CSVs ---
        csv_dir = comp_dir / csv_subdir
        print(f"CSV dir: {csv_dir} {'(existe)' if csv_dir.exists() else '(NO existe)'}")
        if csv_dir.exists():
            for f in sorted(csv_dir.glob("*.csv")):
                try:
                    df = pd.read_csv(f, sep=None, engine="python", encoding="utf-8-sig")
                    dfs[comp]["csv"][f.stem] = df
                    print(f"  ✅ CSV: {f.name} → {df.shape[0]}×{df.shape[1]}")
                except Exception as e:
                    print(f"  ⚠️  CSV fallo {f.name}: {e}")

        # --- Catálogos ---
        cat_dir = comp_dir / cat_subdir
        print(f"CAT dir: {cat_dir} {'(existe)' if cat_dir.exists() else '(NO existe)'}")
        if cat_dir.exists():
            for f in sorted(cat_dir.glob("*.xlsx")):
                try:
                    dv, dval, m, has_vals = read_catalogo(f)
                    dfs[comp]["catalogos"][f.stem] = {"variables": dv, "valores": dval}
                    # fusionar maps por variable (si hay múltiples archivos)
                    for var, kv in m.items():
                        dfs[comp]["maps"].setdefault(var, {}).update(kv)
                    print(f"  📘 CAT: {f.name} | maps añadidos: {len(m)} | valores: {'sí' if has_vals else 'no'}")
                except Exception as e:
                    print(f"  ⚠️  CAT fallo {f.name}: {e}")

        print(f"Resumen {comp}: {len(dfs[comp]['csv'])} CSV | {len(dfs[comp]['catalogos'])} catálogos | {len(dfs[comp]['maps'])} variables con map")

    return dfs

# ---------- EJECUCIÓN MÍNIMA ----------
dfs = cargar_componentes_min(BASE_DIR, COMPONENTES, csv_subdir=CSV_SUBDIR, cat_subdir=CAT_SUBDIR)

# Ejemplos de acceso:
# - Un DataFrame CSV:
#   dfs["nutricion"]["csv"]["rec24h_w"].head()
# - Catálogo (Variables):
#   dfs["salud"]["catalogos"]["adultos_ensanut2023_w_n"]["variables"].head()
# - MAP (código→etiqueta) para una variable:
#   dfs["salud"]["maps"]["resultado_1"]   # dict { '1': 'Completa', '2': 'Incompleta', ... } (ejemplo)



===== COMPONENTE: NUTRICION =====
Base: /content/drive/MyDrive/POSTDOCTORADO C3/ENSANUT/2023/ensanut_2023_ALL/nutricion
CSV dir: /content/drive/MyDrive/POSTDOCTORADO C3/ENSANUT/2023/ensanut_2023_ALL/nutricion/csv (existe)
  ✅ CSV: Antropometria_HTA_4mar24.csv → 3509×97
  ✅ CSV: actividad_fisica_w_adultos.csv → 3977×125
  ✅ CSV: actividad_fisica_w_niños.csv → 568×125
  ✅ CSV: etiquetado_w.csv → 4282×130
  ✅ CSV: frec_ad_rec_w.csv → 311488×20
  ✅ CSV: frec_ad_sup_w.csv → 5952×21
  ✅ CSV: frec_ad_tor_w.csv → 7936×16
  ✅ CSV: frec_adul_w.csv → 1984×54
  ✅ CSV: frec_es_rec_w.csv → 128112×20
  ✅ CSV: frec_es_sup_w.csv → 2448×21
  ✅ CSV: frec_es_tor_w.csv → 3264×16
  ✅ CSV: frec_es_w.csv → 816×52
  ✅ CSV: frec_pr_rec_w.csv → 87848×20
  ✅ CSV: frec_pr_sup_w.csv → 1668×21
  ✅ CSV: frec_pr_tor_w.csv → 2224×16
  ✅ CSV: frec_pr_w.csv → 556×52
  ✅ CSV: lactancia_w.csv → 544×224
  ✅ CSV: plomo_w.csv → 688×91
  ✅ CSV: rec24h_alim_w.csv → 20662×35
  ✅ CSV: rec24h_desc_w.csv → 26714×42
  ✅ CSV: rec24

In [110]:
dfs['nutricion']['csv']['actividad_fisica_w_adultos']
dfs['nutricion']['csv']['actividad_fisica_w_niños']

,FOLIO_INT,FOLIO_I,maquina,hora_ini_1,fecha_ini_1,hora_fin_1,fecha_fin_1,tiempo1,resultado_1,hora_ini_2,...,tiempo,hora_fin,fecha_fin,completa,otroent,ponde_f,estrato,est_sel,upm,x_region
0,2023_01001002_04,2023_01001002,MQ650,21:00:39,04/12/2023,21:04:50,04/12/2023,4,1,,...,4,21:04:50,04/12/2023,1,,"65478,5861070962",3,1000,0100100013312,1
1,2023_02004023_02,2023_02004023,MQ634,17:45:27,04/12/2023,17:50:43,04/12/2023,5,1,,...,5,17:50:43,04/12/2023,1,,"68192,8780830993",3,2000,0200400013107,1
2,2023_03003013_03,2023_03003013,MQ616,15:07:06,07/12/2023,15:13:20,07/12/2023,6,1,,...,6,15:13:20,07/12/2023,1,,"30645,4295275179",3,3000,0300300011998,1
3,2023_04003003_05,2023_04003003,MQ625,10:41:27,10/12/2023,10:45:23,10/12/2023,4,1,,...,4,10:45:23,10/12/2023,1,,"47394,4182209614",3,4000,0400300011155,3
4,2023_04003019_03,2023_04003019,MQ623,12:41:12,10/12/2023,12:45:43,10/12/2023,4,1,,...,4,12:45:43,10/12/2023,1,,"24593,8602660124",3,4000,0400300011155,3
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
563,2023_31017007_04,2023_31017007,MQ623,20:00:57,21/11/2023,20:05:34,21/11/2023,5,1,,...,5,20:05:34,21/11/2023,1,,"27135,5196214077",1,31000,3101700000094,3
564,2023_31017010_04,2023_31017010,MQ625,17:51:43,21/11/2023,17:52:47,21/11/2023,1,2,17:52:58,...,3,17:54:27,21/11/2023,1,,"46401,712005947",1,31000,3101700000094,3
565,2023_31017014_03,2023_31017014,MQ624,14:14:49,21/11/2023,14:18:01,21/11/2023,4,1,,...,4,14:18:01,21/11/2023,1,,"15467,2373353157",1,31000,3101700000094,3
566,2023_32024001_03,2023_32024001,MQ664,18:48:40,07/12/2023,18:53:41,07/12/2023,5,1,,...,5,18:53:41,07/12/2023,1,,"28026,6173484294",2,32000,3202400010322,1


In [134]:
import pandas as pd
df_niños = dfs['nutricion']['csv']['actividad_fisica_w_niños']

In [135]:
import numpy as np
df_niños = df_niños.replace({88: np.nan, 99: np.nan, "88": np.nan, "99": np.nan})
df_niños

,FOLIO_INT,FOLIO_I,maquina,hora_ini_1,fecha_ini_1,hora_fin_1,fecha_fin_1,tiempo1,resultado_1,hora_ini_2,...,tiempo,hora_fin,fecha_fin,completa,otroent,ponde_f,estrato,est_sel,upm,x_region
0,2023_01001002_04,2023_01001002,MQ650,21:00:39,04/12/2023,21:04:50,04/12/2023,4.0,1,,...,4.0,21:04:50,04/12/2023,1,,"65478,5861070962",3,1000,0100100013312,1
1,2023_02004023_02,2023_02004023,MQ634,17:45:27,04/12/2023,17:50:43,04/12/2023,5.0,1,,...,5.0,17:50:43,04/12/2023,1,,"68192,8780830993",3,2000,0200400013107,1
2,2023_03003013_03,2023_03003013,MQ616,15:07:06,07/12/2023,15:13:20,07/12/2023,6.0,1,,...,6.0,15:13:20,07/12/2023,1,,"30645,4295275179",3,3000,0300300011998,1
3,2023_04003003_05,2023_04003003,MQ625,10:41:27,10/12/2023,10:45:23,10/12/2023,4.0,1,,...,4.0,10:45:23,10/12/2023,1,,"47394,4182209614",3,4000,0400300011155,3
4,2023_04003019_03,2023_04003019,MQ623,12:41:12,10/12/2023,12:45:43,10/12/2023,4.0,1,,...,4.0,12:45:43,10/12/2023,1,,"24593,8602660124",3,4000,0400300011155,3
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
563,2023_31017007_04,2023_31017007,MQ623,20:00:57,21/11/2023,20:05:34,21/11/2023,5.0,1,,...,5.0,20:05:34,21/11/2023,1,,"27135,5196214077",1,31000,3101700000094,3
564,2023_31017010_04,2023_31017010,MQ625,17:51:43,21/11/2023,17:52:47,21/11/2023,1.0,2,17:52:58,...,3.0,17:54:27,21/11/2023,1,,"46401,712005947",1,31000,3101700000094,3
565,2023_31017014_03,2023_31017014,MQ624,14:14:49,21/11/2023,14:18:01,21/11/2023,4.0,1,,...,4.0,14:18:01,21/11/2023,1,,"15467,2373353157",1,31000,3101700000094,3
566,2023_32024001_03,2023_32024001,MQ664,18:48:40,07/12/2023,18:53:41,07/12/2023,5.0,1,,...,5.0,18:53:41,07/12/2023,1,,"28026,6173484294",2,32000,3202400010322,1


In [136]:
for i in df_niños.columns:
    print(i)

FOLIO_INT
FOLIO_I
maquina
hora_ini_1
fecha_ini_1
hora_fin_1
fecha_fin_1
tiempo1
resultado_1
hora_ini_2
fecha_ini_2
hora_fin_2
fecha_fin_2
tiempo2
resultado_2
hora_ini_3
fecha_ini_3
hora_fin_3
fecha_fin_3
tiempo3
resultado_3
hora_ini_4
fecha_ini_4
hora_fin_4
fecha_fin_4
tiempo4
resultado_4
hora_ini
fecha_ini
entidad
desc_ent
municipio
desc_mun
nota1
h0302
h0303
fech_nac
nota
fd0400a
fd0400b
fd0401a
fd0401b
fd0402
fd0403
fd0404
fd0405
fd0406
fd0407
fd04vd
fd0408
fd0409
fd0409e
fd0500
FD0501A
FD0501B
FD0501C
FD0501D
FD0501E1
FD0501F
FD0501G
FD0501H
FD0501I
FD0501J
FD0501K
FD0501L
FD0501M
FD0501N
FD0501O
FD0501P
FD0501Q
FD0501R
FD0501S
FD0501T
fd0501e
nota2
fd0502
fd0503
nota3
fa0400
nota4
fa0401
fa0402ah
fa0402am
fa0402bh
fa0402bm
nota5
fa0403
fa0404ah
fa0404am
fa0404bh
fa0404bm
nota6
fa0405
fa0406ah
fa0406am
fa0406bh
fa0406bm
nota7
fa0407h
fa0407m
fa0407ah
fa0407am
nota8
fa0408
fa0409h
fa0409m
nota9
fa0410
fa0411
fa0412
fa0413
fa0414
fa0415
fa0416
comentario
tiempo
hora_fin
fecha_fin
com

#limpieza

In [138]:
df_niños[['fd0400a', 'fd0400b', 'fd0401a', 'fd0401b', 'fd0402', 'fd0403', 'fd0404', 'fd0405', 'fd0406', 'fd0407', 'fd04vd', 'fd0408', 'fd0409', 'fd0500', 'FD0501A', 'FD0501B', 'FD0501C', 'FD0501D', 'FD0501E1', 'FD0501F', 'FD0501G', 'FD0501H', 'FD0501I', 'FD0501J', 'FD0501K', 'FD0501L', 'FD0501M', 'FD0501N', 'FD0501O', 'FD0501P', 'FD0501Q', 'FD0501R', 'FD0501S', 'FD0501T', 'fd0502', 'fd0503']]

,fd0400a,fd0400b,fd0401a,fd0401b,fd0402,fd0403,fd0404,fd0405,fd0406,fd0407,...,FD0501M,FD0501N,FD0501O,FD0501P,FD0501Q,FD0501R,FD0501S,FD0501T,fd0502,fd0503
0,7.0,7.0,7.0,7.0,1.0,1.0,1.0,1.0,2.0,3.0,...,,,,,,,,,1,1
1,7.0,7.0,6.0,6.0,2.0,1.0,3.0,4.0,2.0,2.0,...,,,,,,,,,5,5
2,5.0,7.0,2.0,5.0,2.0,3.0,1.0,3.0,0.0,0.0,...,,,,,,,,,4,2
3,7.0,7.0,5.0,5.0,6.0,2.0,0.0,0.0,2.0,4.0,...,,,,,,,,,6,1
4,6.0,6.0,5.0,5.0,2.0,0.0,2.0,2.0,2.0,2.0,...,,,,,,,,,2,2
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
563,6.0,2.0,2.0,3.0,1.0,1.0,2.0,2.0,1.0,1.0,...,,,,,,,,,2,5
564,5.0,4.0,5.0,5.0,2.0,3.0,2.0,2.0,2.0,2.0,...,,,,,,,,,3,3
565,7.0,7.0,3.0,3.0,2.0,2.0,1.0,1.0,0.0,0.0,...,,,,,,,,,0,0
566,6.0,7.0,2.0,6.0,1.0,2.0,1.0,1.0,2.0,2.0,...,,,,,,,,,4,5


In [139]:
variables_actividad_fisica_niños.keys()

dict_keys(['fd0400a', 'fd0400b', 'fd0401a', 'fd0401b', 'fd0402', 'fd0403', 'fd0404', 'fd0405', 'fd0406', 'fd0407', 'fd04vd', 'fd0408', 'fd0409', 'fd0500', 'FD0501A', 'FD0501B', 'FD0501C', 'FD0501D', 'FD0501E1', 'FD0501F', 'FD0501G', 'FD0501H', 'FD0501I', 'FD0501J', 'FD0501K', 'FD0501L', 'FD0501M', 'FD0501N', 'FD0501O', 'FD0501P', 'FD0501Q', 'FD0501R', 'FD0501S', 'FD0501T', 'fd0502', 'fd0503'])

In [140]:

diccionario_tipo_de_actividades=  {
    1:  "Artes marciales (karate, Tae Kwon do, jujitsu, etc.)",
    2:  "Bádminton",
    3:  "Bailar (ballet, jazz, etc.)",
    4:  "Básquetbol",
    5:  "Béisbol o softbol",
    6:  "Bicicleta",
    7:  "Boxeo",
    8:  "Caminar",
    9:  "Correr",
    10: "Frontón",
    11: "Fútbol",
    12: "Gimnasia",
    13: "Natación",
    14: "Patinar o andar en patineta",
    15: "Ping pong",
    16: "Tenis",
    17: "Voleibol",
    18: "Otra, ¿Cuál?"

}

variables_actividad_fisica_niños = {
    "fd0400a": {
        "Descripcion": "Hora a la que el niño se duerme entre semana.",
        "Diccionario": {
            1: "Antes de las 6",
            2: "Entre 6 y 7",
            3: "Entre 7 y 8",
            4: "Entre 8 y 9",
            5: "Entre 9 y 10",
            6: "Entre 10 y 11",
            7: "Después de las 11"
        }
    },
    "fd0400b": {
        "Descripcion": "Hora a la que el niño se duerme en fin de semana.",
        "Diccionario": {
            1: "Antes de las 6",
            2: "Entre 6 y 7",
            3: "Entre 7 y 8",
            4: "Entre 8 y 9",
            5: "Entre 9 y 10",
            6: "Entre 10 y 11",
            7: "Después de las 11",

        }
    },
    "fd0401a": {
        "Descripcion": "Hora a la que el niño se levanta entre semana.",
        "Diccionario": {
            1: "Antes de las 6",
            2: "Entre 6 y 7",
            3: "Entre 7 y 8",
            4: "Entre 8 y 9",
            5: "Entre 9 y 10",
            6: "Entre 10 y 11",
            7: "Después de las 11"
        }
    },
    "fd0401b": {
        "Descripcion": "Hora a la que el niño se levanta en fin de semana.",
        "Diccionario": {
            1: "Antes de las 6",
            2: "Entre 6 y 7",
            3: "Entre 7 y 8",
            4: "Entre 8 y 9",
            5: "Entre 9 y 10",
            6: "Entre 10 y 11",
            7: "Después de las 11",
        }
    },
    "fd0402": {
        "Descripcion": "Horas de televisión entre semana.",
        "Diccionario": {
            0: "Nada",
            1: "Menos de una hora",
            2: "1-2 horas",
            3: "3-4 horas",
            4: "5-6 horas",
            5: "7-8 horas",
            6: "9 o más horas",
            8: "No responde",

        }
    },
    "fd0403": {
        "Descripcion": "Horas de televisión en fin de semana.",
        "Diccionario": {
            0: "Nada",
            1: "Menos de una hora",
            2: "1-2 horas",
            3: "3-4 horas",
            4: "5-6 horas",
            5: "7-8 horas",
            6: "9 o más horas",

        }
    },
    "fd0404": {
        "Descripcion": "Horas de videojuegos entre semana.",
        "Diccionario": {
            0: "Nada",
            1: "Menos de una hora",
            2: "1-2 horas",
            3: "3-4 horas",
            4: "5-6 horas",
            5: "7-8 horas",
            6: "9 o más horas"
        }
    },
    "fd0405": {
        "Descripcion": "Horas de videojuegos en fin de semana.",
        "Diccionario": {
            0: "Nada",
            1: "Menos de una hora",
            2: "1-2 horas",
            3: "3-4 horas",
            4: "5-6 horas",
            5: "7-8 horas",
            6: "9 o más horas"

        }
    },
    "fd0406": {
        "Descripcion": "Horas frente a computadora/tableta/teléfono entre semana (no videojuegos).",
        "Diccionario": {
            0: "Nada",
            1: "Menos de una hora",
            2: "1-2 horas",
            3: "3-4 horas",
            4: "5-6 horas",
            5: "7-8 horas",
            6: "9 o más horas"
        }
    },
    "fd0407": {
        "Descripcion": "Horas frente a computadora/tableta/teléfono fin de semana (no videojuegos).",
        "Diccionario": {
            0: "Nada",
            1: "Menos de una hora",
            2: "1-2 horas",
            3: "3-4 horas",
            4: "5-6 horas",
            5: "7-8 horas",
            6: "9 o más horas",

        }
    },
    "fd04vd": {
        "Descripcion": "El menor tiene alguna discapacidad física que le impida moverse para realizar actividad física.",
        "Diccionario": {
            1: "Sí",
            2: "No"
        }
    },
    "fd0408": {
        "Descripcion": "Tiempo de traslado de casa a escuela en un día típico.",
        "Diccionario": {
            1: "Menos de 10 minutos",
            2: "Entre 10 y 30 minutos",
            3: "Entre 30 minutos y 1 hora",
            4: "Entre 1 y 2 horas",
            5: "Entre 2 y 3 horas",
            6: "Más de 3 horas",
            55: "No va a la escuela"
        }
    },
    "fd0409": {
        "Descripcion": "Medio de transporte principal a la escuela.",
        "Diccionario": {
            1: "Caminata",
            2: "Bicicleta pedaleada por el niño",
            3: "Bicicleta pedaleada por alguien más",
            4: "Autobús/tren/tranvía/metro/colectivo/bote",
            5: "Carro/motocicleta/motoneta",
            6: "Otro",
            55: "No va a la escuela",

        }
    },
    "fd0500": {
        "Descripcion": "Número de actividades físicas o deportivas en que participó el último año.",
        "Diccionario": {
            0: "Ninguna",
            1: "1 actividad",
            2: "2 actividades",
            3: "3 actividades",
            4: "4 o más actividades",

        }
    },
    "FD0501A": {"Descripcion": "Practicó artes marciales (karate, tae kwon do, jujitsu, etc.)", "Diccionario": diccionario_tipo_de_actividades},
    "FD0501B": {"Descripcion": "Practicó bádminton", "Diccionario": diccionario_tipo_de_actividades},
    "FD0501C": {"Descripcion": "Practicó baile (ballet, jazz, etc.)", "Diccionario": diccionario_tipo_de_actividades},
    "FD0501D": {"Descripcion": "Practicó básquetbol", "Diccionario": diccionario_tipo_de_actividades},
    "FD0501E1": {"Descripcion": "Practicó béisbol o softbol", "Diccionario": diccionario_tipo_de_actividades},
    "FD0501F": {"Descripcion": "Practicó bicicleta", "Diccionario": diccionario_tipo_de_actividades},
    "FD0501G": {"Descripcion": "Practicó boxeo", "Diccionario": diccionario_tipo_de_actividades},
    "FD0501H": {"Descripcion": "Practicó caminar", "Diccionario": diccionario_tipo_de_actividades},
    "FD0501I": {"Descripcion": "Practicó correr", "Diccionario": diccionario_tipo_de_actividades},
    "FD0501J": {"Descripcion": "Practicó frontón", "Diccionario": diccionario_tipo_de_actividades},
    "FD0501K": {"Descripcion": "Practicó fútbol", "Diccionario": diccionario_tipo_de_actividades},
    "FD0501L": {"Descripcion": "Practicó gimnasia", "Diccionario": diccionario_tipo_de_actividades},
    "FD0501M": {"Descripcion": "Practicó natación", "Diccionario": diccionario_tipo_de_actividades},
    "FD0501N": {"Descripcion": "Practicó patinar o andar en patineta", "Diccionario": diccionario_tipo_de_actividades},
    "FD0501O": {"Descripcion": "Practicó ping pong", "Diccionario": {}},
    "FD0501P": {"Descripcion": "Practicó tenis", "Diccionario": {}},
    "FD0501Q": {"Descripcion": "Practicó voleibol", "Diccionario": {}},
    "FD0501R": {"Descripcion": "Practicó otra actividad física (especificar)", "Diccionario": {}},
    "FD0501S": {"Descripcion": "Practicó otra actividad física (campo extra)", "Diccionario": {}},
    "FD0501T": {"Descripcion": "Practicó otra actividad física (campo extra)", "Diccionario": {}},

    "fd0502": {
        "Descripcion": "Número de días activos ≥60 min en los últimos 7 días.",
        "Diccionario": {
            0: "0 días",
            1: "1 día",
            2: "2 días",
            3: "3 días",
            4: "4 días",
            5: "5 días",
            6: "6 días",
            7: "7 días",

        }
    },
    "fd0503": {
        "Descripcion": "Número de días activos ≥60 min en una semana típica.",
        "Diccionario": {
            0: "0 días",
            1: "1 día",
            2: "2 días",
            3: "3 días",
            4: "4 días",
            5: "5 días",
            6: "6 días",
            7: "7 días",

        }
    }
}


In [141]:
variables_actividad_fisica_niños.keys()

dict_keys(['fd0400a', 'fd0400b', 'fd0401a', 'fd0401b', 'fd0402', 'fd0403', 'fd0404', 'fd0405', 'fd0406', 'fd0407', 'fd04vd', 'fd0408', 'fd0409', 'fd0500', 'FD0501A', 'FD0501B', 'FD0501C', 'FD0501D', 'FD0501E1', 'FD0501F', 'FD0501G', 'FD0501H', 'FD0501I', 'FD0501J', 'FD0501K', 'FD0501L', 'FD0501M', 'FD0501N', 'FD0501O', 'FD0501P', 'FD0501Q', 'FD0501R', 'FD0501S', 'FD0501T', 'fd0502', 'fd0503'])

In [142]:
import pandas as pd
import numpy as np

# --- 1) Reemplazar "88"/"99" por NaN antes de convertir ---
df_niños = df_niños.replace({"88": np.nan, "99": np.nan})

# --- 2) Convertir a número las columnas que vamos a mapear ---
for var in variables_actividad_fisica_niños.keys():
    if var in df_niños.columns:
        # convierte "01" -> 1, mantiene NaN
        df_niños[var] = pd.to_numeric(df_niños[var], errors="coerce")

# --- 3) Mapear usando el diccionario ---
for var, info in variables_actividad_fisica_niños.items():
    if var in df_niños.columns:
        dic = info.get("Diccionario", {})
        if dic:
            df_niños[var] = df_niños[var].map(dic)

# --- 4) Revisar value_counts ---
for var, info in variables_actividad_fisica_niños.items():
    if var in df_niños.columns:
        print(f"\n=== {var} ({info['Descripcion']}) ===")
        print(df_niños[var].value_counts(dropna=False))





=== fd0400a (Hora a la que el niño se duerme entre semana.) ===
fd0400a
Entre 9 y 10         182
Entre 10 y 11        177
Después de las 11    104
Entre 8 y 9           78
Entre 7 y 8           17
Entre 6 y 7            9
NaN                    1
Name: count, dtype: int64

=== fd0400b (Hora a la que el niño se duerme en fin de semana.) ===
fd0400b
Después de las 11    219
Entre 10 y 11        164
Entre 9 y 10         119
Entre 8 y 9           47
Entre 7 y 8           11
Entre 6 y 7            7
NaN                    1
Name: count, dtype: int64

=== fd0401a (Hora a la que el niño se levanta entre semana.) ===
fd0401a
Entre 6 y 7          238
Antes de las 6       107
Entre 7 y 8           94
Entre 8 y 9           53
Entre 9 y 10          41
Entre 10 y 11         23
Después de las 11     11
NaN                    1
Name: count, dtype: int64

=== fd0401b (Hora a la que el niño se levanta en fin de semana.) ===
fd0401b
Entre 8 y 9          131
Entre 7 y 8          121
Entre 9 y 10        

In [143]:
df_niños[['fd0400a', 'fd0400b', 'fd0401a', 'fd0401b', 'fd0402', 'fd0403', 'fd0404', 'fd0405', 'fd0406', 'fd0407', 'fd04vd', 'fd0408', 'fd0409', 'fd0500', 'FD0501A', 'FD0501B', 'FD0501C', 'FD0501D', 'FD0501E1', 'FD0501F', 'FD0501G', 'FD0501H', 'FD0501I', 'FD0501J', 'FD0501K', 'FD0501L', 'FD0501M', 'FD0501N', 'FD0501O', 'FD0501P', 'FD0501Q', 'FD0501R', 'FD0501S', 'FD0501T', 'fd0502', 'fd0503']]

,fd0400a,fd0400b,fd0401a,fd0401b,fd0402,fd0403,fd0404,fd0405,fd0406,fd0407,...,FD0501M,FD0501N,FD0501O,FD0501P,FD0501Q,FD0501R,FD0501S,FD0501T,fd0502,fd0503
0,Después de las 11,Después de las 11,Después de las 11,Después de las 11,Menos de una hora,Menos de una hora,Menos de una hora,Menos de una hora,1-2 horas,3-4 horas,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,1 día,1 día
1,Después de las 11,Después de las 11,Entre 10 y 11,Entre 10 y 11,1-2 horas,Menos de una hora,3-4 horas,5-6 horas,1-2 horas,1-2 horas,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,5 días,5 días
2,Entre 9 y 10,Después de las 11,Entre 6 y 7,Entre 9 y 10,1-2 horas,3-4 horas,Menos de una hora,3-4 horas,Nada,Nada,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,4 días,2 días
3,Después de las 11,Después de las 11,Entre 9 y 10,Entre 9 y 10,9 o más horas,1-2 horas,Nada,Nada,1-2 horas,5-6 horas,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,6 días,1 día
4,Entre 10 y 11,Entre 10 y 11,Entre 9 y 10,Entre 9 y 10,1-2 horas,Nada,1-2 horas,1-2 horas,1-2 horas,1-2 horas,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,2 días,2 días
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
563,Entre 10 y 11,Entre 6 y 7,Entre 6 y 7,Entre 7 y 8,Menos de una hora,Menos de una hora,1-2 horas,1-2 horas,Menos de una hora,Menos de una hora,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,2 días,5 días
564,Entre 9 y 10,Entre 8 y 9,Entre 9 y 10,Entre 9 y 10,1-2 horas,3-4 horas,1-2 horas,1-2 horas,1-2 horas,1-2 horas,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,3 días,3 días
565,Después de las 11,Después de las 11,Entre 7 y 8,Entre 7 y 8,1-2 horas,1-2 horas,Menos de una hora,Menos de una hora,Nada,Nada,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0 días,0 días
566,Entre 10 y 11,Después de las 11,Entre 6 y 7,Entre 10 y 11,Menos de una hora,1-2 horas,Menos de una hora,Menos de una hora,1-2 horas,1-2 horas,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,4 días,5 días


#se logro!

In [180]:
import numpy as np
df_niños = dfs['nutricion']['csv']['actividad_fisica_w_niños']
df_niños = df_niños[ ['FOLIO_INT', 'FOLIO_I'] + list(variables_actividad_fisica_niños.keys()) ]

df_niños = df_niños.replace({88: np.nan, 99: np.nan, "88": np.nan, "99": np.nan})
df_niños

,FOLIO_INT,FOLIO_I,fd0400a,fd0400b,fd0401a,fd0401b,fd0402,fd0403,fd0404,fd0405,...,FD0501M,FD0501N,FD0501O,FD0501P,FD0501Q,FD0501R,FD0501S,FD0501T,fd0502,fd0503
0,2023_01001002_04,2023_01001002,7.0,7.0,7.0,7.0,1.0,1.0,1.0,1.0,...,,,,,,,,,1,1
1,2023_02004023_02,2023_02004023,7.0,7.0,6.0,6.0,2.0,1.0,3.0,4.0,...,,,,,,,,,5,5
2,2023_03003013_03,2023_03003013,5.0,7.0,2.0,5.0,2.0,3.0,1.0,3.0,...,,,,,,,,,4,2
3,2023_04003003_05,2023_04003003,7.0,7.0,5.0,5.0,6.0,2.0,0.0,0.0,...,,,,,,,,,6,1
4,2023_04003019_03,2023_04003019,6.0,6.0,5.0,5.0,2.0,0.0,2.0,2.0,...,,,,,,,,,2,2
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
563,2023_31017007_04,2023_31017007,6.0,2.0,2.0,3.0,1.0,1.0,2.0,2.0,...,,,,,,,,,2,5
564,2023_31017010_04,2023_31017010,5.0,4.0,5.0,5.0,2.0,3.0,2.0,2.0,...,,,,,,,,,3,3
565,2023_31017014_03,2023_31017014,7.0,7.0,3.0,3.0,2.0,2.0,1.0,1.0,...,,,,,,,,,0,0
566,2023_32024001_03,2023_32024001,6.0,7.0,2.0,6.0,1.0,2.0,1.0,1.0,...,,,,,,,,,4,5


#ADULTOS

In [181]:
df_adultos = dfs['nutricion']['csv']['actividad_fisica_w_adultos']
df_adultos.head()

,FOLIO_INT,FOLIO_I,maquina,hora_ini_1,fecha_ini_1,hora_fin_1,fecha_fin_1,tiempo1,resultado_1,hora_ini_2,...,tiempo,hora_fin,fecha_fin,completa,otroent,ponde_f,estrato,est_sel,upm,x_region
0,2023_01001001_01,2023_01001001,MQ650,20:01:57,04/12/2023,20:05:32,04/12/2023,4,1,,...,4,20:05:32,04/12/2023,1,,"21300,7807393647",3,1000,0100100013312,1
1,2023_01001001_02,2023_01001001,MQ650,14:06:05,04/12/2023,14:13:23,04/12/2023,7,1,,...,7,14:13:23,04/12/2023,1,,"60899,5631254304",3,1000,0100100013312,1
2,2023_01001001_04,2023_01001001,MQ650,14:16:38,04/12/2023,14:21:46,04/12/2023,5,1,,...,5,14:21:46,04/12/2023,1,,"53405,8012888781",3,1000,0100100013312,1
3,2023_01001002_01,2023_01001002,MQ650,20:55:25,04/12/2023,21:00:19,04/12/2023,5,1,,...,5,21:00:19,04/12/2023,1,,"30449,7815627152",3,1000,0100100013312,1
4,2023_01001003_01,2023_01001003,MQ650,12:30:33,04/12/2023,12:36:35,04/12/2023,6,1,,...,6,12:36:35,04/12/2023,1,,"21300,7807393647",3,1000,0100100013312,1


In [182]:
df_adultos['fa0403'].value_counts()# es de control pero no deberia ir a la lista

,count
fa0403,
0,1519
7,1127
2,308
3,287
1,218
5,212
4,151
6,144
,8


In [183]:
variables_actividad_fisica_adultos = {
    # Sueño
    "fa0400": {
        "Descripcion": "Horas que duerme en promedio en un día.",
        "Diccionario": {}  # Es numérica abierta (horas), no hay categorías
    },

    # Actividad física vigorosa
    "fa0401": {
        "Descripcion": "Días en los últimos 7 días que realizó actividad física vigorosa.",
        "Diccionario": {
            0: "No realiza actividad vigorosa",
            55: "Imposibilidad para moverse o caminar"
        }
    },
    "fa0402ah": {
        "Descripcion": "Horas por día de actividad vigorosa (en uno de esos días).",
        "Diccionario": {}  # numérico libre
    },
    "fa0402am": {
        "Descripcion": "Minutos por día de actividad vigorosa (en uno de esos días).",
        "Diccionario": {}
    },
    "fa0402bh": {
        "Descripcion": "Horas totales de actividad vigorosa en los últimos 7 días.",
        "Diccionario": {}
    },
    "fa0402bm": {
        "Descripcion": "Minutos totales de actividad vigorosa en los últimos 7 días.",
        "Diccionario": {}
    },

    # Actividad física moderada
    "fa0403": {
        "Descripcion": "Días en los últimos 7 días que realizó actividad física moderada.",
        "Diccionario": {
            0: "No realiza actividad moderada",

        }
    },
    "fa0404ah": {
        "Descripcion": "Horas por día de actividad moderada (en uno de esos días).",
        "Diccionario": {}
    },
    "fa0404am": {
        "Descripcion": "Minutos por día de actividad moderada (en uno de esos días).",
        "Diccionario": {}
    },
    "fa0404bh": {
        "Descripcion": "Horas totales de actividad moderada en los últimos 7 días.",
        "Diccionario": {}
    },
    "fa0404bm": {
        "Descripcion": "Minutos totales de actividad moderada en los últimos 7 días.",
        "Diccionario": {}
    },

    # Caminata
    "fa0405": {
        "Descripcion": "Días en los últimos 7 días que caminó ≥10 min continuos.",
        "Diccionario": {
            0: "No caminó",
            88: "No responde",
            99: "No sabe"
        }
    },
    "fa0406ah": {
        "Descripcion": "Horas por día que caminó (en uno de esos días).",
        "Diccionario": {}
    },
    "fa0406am": {
        "Descripcion": "Minutos por día que caminó (en uno de esos días).",
        "Diccionario": {}
    },
    "fa0406bh": {
        "Descripcion": "Horas totales caminadas en los últimos 7 días.",
        "Diccionario": {}
    },
    "fa0406bm": {
        "Descripcion": "Minutos totales caminados en los últimos 7 días.",
        "Diccionario": {}
    },

    # Tiempo sentado
    "fa0407h": {
        "Descripcion": "Horas sentado en un día típico de semana pasada.",
        "Diccionario": {}
    },
    "fa0407m": {
        "Descripcion": "Minutos sentado en un día típico de semana pasada.",
        "Diccionario": {}
    },
    "fa0407ah": {
        "Descripcion": "Horas sentado el miércoles pasado.",
        "Diccionario": {}
    },
    "fa0407am": {
        "Descripcion": "Minutos sentado el miércoles pasado.",
        "Diccionario": {}
    },

    # Transporte
    "fa0408": {
        "Descripcion": "Días que se desplazó en vehículo motorizado últimos 7 días.",
        "Diccionario": {
            0: "No viajó en vehículo",
            88: "No responde",
            99: "No sabe"
        }
    },
    "fa0409h": {
        "Descripcion": "Horas viajando en vehículo en un día típico.",
        "Diccionario": {}
    },
    "fa0409m": {
        "Descripcion": "Minutos viajando en vehículo en un día típico.",
        "Diccionario": {}
    },

    # Pantallas - TV
    "fa0410": {
        "Descripcion": "Horas frente a TV entre semana.",
        "Diccionario": {
            0: "Nada",
            1: "Menos de una hora",
            2: "1-2 horas",
            3: "3-4 horas",
            4: "5-6 horas",
            5: "7-8 horas",
            6: "9 o más horas",
            88: "No responde",
            99: "No sabe"
        }
    },
    "fa0411": {
        "Descripcion": "Horas frente a TV fin de semana.",
        "Diccionario": {
            0: "Nada",
            1: "Menos de una hora",
            2: "1-2 horas",
            3: "3-4 horas",
            4: "5-6 horas",
            5: "7-8 horas",
            6: "9 o más horas",
            88: "No responde",
            99: "No sabe"
        }
    },

    # Pantallas - Videojuegos
    "fa0412": {
        "Descripcion": "Horas jugando videojuegos entre semana.",
        "Diccionario": {
            0: "Nada",
            1: "Menos de una hora",
            2: "1-2 horas",
            3: "3-4 horas",
            4: "5-6 horas",
            5: "7-8 horas",
            6: "9 o más horas",
            88: "No responde",
            99: "No sabe"
        }
    },
    "fa0413": {
        "Descripcion": "Horas jugando videojuegos fin de semana.",
        "Diccionario": {
            0: "Nada",
            1: "Menos de una hora",
            2: "1-2 horas",
            3: "3-4 horas",
            4: "5-6 horas",
            5: "7-8 horas",
            6: "9 o más horas",
            88: "No responde",
            99: "No sabe"
        }
    },

    # Pantallas - Computadora/Internet
    "fa0414": {
        "Descripcion": "Horas usando computadora/internet entre semana.",
        "Diccionario": {
            0: "Nada",
            1: "Menos de una hora",
            2: "1-2 horas",
            3: "3-4 horas",
            4: "5-6 horas",
            5: "7-8 horas",
            6: "9 o más horas",
            88: "No responde",
            99: "No sabe"
        }
    },
    "fa0415": {
        "Descripcion": "Horas usando computadora/internet fin de semana.",
        "Diccionario": {
            0: "Nada",
            1: "Menos de una hora",
            2: "1-2 horas",
            3: "3-4 horas",
            4: "5-6 horas",
            5: "7-8 horas",
            6: "9 o más horas",
            88: "No responde",
            99: "No sabe"
        }
    },

    # Cambio últimos 3 meses
    "fa0416": {
        "Descripcion": "Comparación de actividad física semana pasada vs últimos 3 meses.",
        "Diccionario": {
            0: "Más",
            1: "Menos",
            2: "Más o menos igual",
            88: "No responde",
            99: "No sabe"
        }
    }
}


In [184]:
df_adultos[list(variables_actividad_fisica_adultos.keys())]

,fa0400,fa0401,fa0402ah,fa0402am,fa0402bh,fa0402bm,fa0403,fa0404ah,fa0404am,fa0404bh,...,fa0408,fa0409h,fa0409m,fa0410,fa0411,fa0412,fa0413,fa0414,fa0415,fa0416
0,2,0,,,,,0,,,,...,4,0,30,1,2,0,0,1,2,2
1,1,0,,,,,0,,,,...,7,0,40,1,1,0,0,4,4,1
2,5,0,,,,,0,,,,...,7,1,0,1,1,0,0,4,4,2
3,1,0,,,,,0,,,,...,7,3,0,1,1,0,0,2,2,0
4,3,0,,,,,0,,,,...,3,1,0,2,3,0,0,1,0,2
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
3972,4,0,,,,,0,,,,...,7,0,20,2,2,0,0,2,2,0
3973,3,0,,,,,0,,,,...,3,6,0,3,4,0,0,5,4,2
3974,3,0,,,,,0,,,,...,0,,,2,2,0,0,0,0,2
3975,3,0,,,,,7,5,0,,...,7,0,30,2,3,0,0,2,2,2


In [185]:
import pandas as pd
import numpy as np

# --- 1) Reemplazar "88"/"99" por NaN antes de convertir ---
df_adultos = df_adultos.replace({"88": np.nan, "99": np.nan})

# --- 2) Convertir a número las columnas que vamos a mapear ---
for var in variables_actividad_fisica_adultos.keys():
    if var in df_adultos.columns:
        df_adultos[var] = pd.to_numeric(df_adultos[var], errors="coerce")

# --- 3) Mapear usando el diccionario ---
for var, info in variables_actividad_fisica_adultos.items():
    if var in df_adultos.columns:
        dic = info.get("Diccionario", {})
        if dic:
            df_adultos[var] = df_adultos[var].map(dic)

# --- 4) Revisar value_counts ---
for var, info in variables_actividad_fisica_adultos.items():
    if var in df_adultos.columns:
        print(f"\n=== {var} ({info['Descripcion']}) ===")
        print(df_adultos[var].value_counts(dropna=False))



=== fa0400 (Horas que duerme en promedio en un día.) ===
fa0400
4     1386
3     1001
2      792
5      451
1      327
99      18
88       2
Name: count, dtype: int64

=== fa0401 (Días en los últimos 7 días que realizó actividad física vigorosa.) ===
fa0401
No realiza actividad vigorosa           2592
NaN                                     1377
Imposibilidad para moverse o caminar       8
Name: count, dtype: int64

=== fa0402ah (Horas por día de actividad vigorosa (en uno de esos días).) ===
fa0402ah
NaN     2605
0.0      351
1.0      294
2.0      195
3.0      107
8.0       99
6.0       70
5.0       63
4.0       59
7.0       45
10.0      35
9.0       32
12.0      15
11.0       4
14.0       1
13.0       1
15.0       1
Name: count, dtype: int64

=== fa0402am (Minutos por día de actividad vigorosa (en uno de esos días).) ===
fa0402am
NaN     2605
0.0      891
30.0     246
20.0      73
10.0      49
40.0      34
15.0      34
45.0      13
5.0        6
50.0       6
1.0        4
2.0        4

In [186]:
df_adultos[variables_actividad_fisica_adultos.keys()]

,fa0400,fa0401,fa0402ah,fa0402am,fa0402bh,fa0402bm,fa0403,fa0404ah,fa0404am,fa0404bh,...,fa0408,fa0409h,fa0409m,fa0410,fa0411,fa0412,fa0413,fa0414,fa0415,fa0416
0,2,No realiza actividad vigorosa,NaN,NaN,NaN,NaN,No realiza actividad moderada,NaN,NaN,NaN,...,NaN,0.0,30.0,Menos de una hora,1-2 horas,Nada,Nada,Menos de una hora,1-2 horas,Más o menos igual
1,1,No realiza actividad vigorosa,NaN,NaN,NaN,NaN,No realiza actividad moderada,NaN,NaN,NaN,...,NaN,0.0,40.0,Menos de una hora,Menos de una hora,Nada,Nada,5-6 horas,5-6 horas,Menos
2,5,No realiza actividad vigorosa,NaN,NaN,NaN,NaN,No realiza actividad moderada,NaN,NaN,NaN,...,NaN,1.0,0.0,Menos de una hora,Menos de una hora,Nada,Nada,5-6 horas,5-6 horas,Más o menos igual
3,1,No realiza actividad vigorosa,NaN,NaN,NaN,NaN,No realiza actividad moderada,NaN,NaN,NaN,...,NaN,3.0,0.0,Menos de una hora,Menos de una hora,Nada,Nada,1-2 horas,1-2 horas,Más
4,3,No realiza actividad vigorosa,NaN,NaN,NaN,NaN,No realiza actividad moderada,NaN,NaN,NaN,...,NaN,1.0,0.0,1-2 horas,3-4 horas,Nada,Nada,Menos de una hora,Nada,Más o menos igual
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
3972,4,No realiza actividad vigorosa,NaN,NaN,NaN,NaN,No realiza actividad moderada,NaN,NaN,NaN,...,NaN,0.0,20.0,1-2 horas,1-2 horas,Nada,Nada,1-2 horas,1-2 horas,Más
3973,3,No realiza actividad vigorosa,NaN,NaN,NaN,NaN,No realiza actividad moderada,NaN,NaN,NaN,...,NaN,6.0,0.0,3-4 horas,5-6 horas,Nada,Nada,7-8 horas,5-6 horas,Más o menos igual
3974,3,No realiza actividad vigorosa,NaN,NaN,NaN,NaN,No realiza actividad moderada,NaN,NaN,NaN,...,No viajó en vehículo,NaN,NaN,1-2 horas,1-2 horas,Nada,Nada,Nada,Nada,Más o menos igual
3975,3,No realiza actividad vigorosa,NaN,NaN,NaN,NaN,NaN,5.0,0.0,NaN,...,NaN,0.0,30.0,1-2 horas,3-4 horas,Nada,Nada,1-2 horas,1-2 horas,Más o menos igual


In [189]:
import numpy as np
df_adultos = dfs['nutricion']['csv']['actividad_fisica_w_adultos']

df_adultos = df_adultos[ ['FOLIO_INT', 'FOLIO_I'] + list(variables_actividad_fisica_adultos.keys()) ]

df_adultos = df_adultos.replace({88: np.nan, 99: np.nan, "88": np.nan, "99": np.nan})
df_adultos
df_adultos

,FOLIO_INT,FOLIO_I,fa0400,fa0401,fa0402ah,fa0402am,fa0402bh,fa0402bm,fa0403,fa0404ah,...,fa0408,fa0409h,fa0409m,fa0410,fa0411,fa0412,fa0413,fa0414,fa0415,fa0416
0,2023_01001001_01,2023_01001001,2.0,0.0,,,,,0,,...,4,0,30,1,2,0,0,1,2,2
1,2023_01001001_02,2023_01001001,1.0,0.0,,,,,0,,...,7,0,40,1,1,0,0,4,4,1
2,2023_01001001_04,2023_01001001,5.0,0.0,,,,,0,,...,7,1,0,1,1,0,0,4,4,2
3,2023_01001002_01,2023_01001002,1.0,0.0,,,,,0,,...,7,3,0,1,1,0,0,2,2,0
4,2023_01001003_01,2023_01001003,3.0,0.0,,,,,0,,...,3,1,0,2,3,0,0,1,0,2
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
3972,2023_32024023_01,2023_32024023,4.0,0.0,,,,,0,,...,7,0,20,2,2,0,0,2,2,0
3973,2023_32024024_01,2023_32024024,3.0,0.0,,,,,0,,...,3,6,0,3,4,0,0,5,4,2
3974,2023_32024026_02,2023_32024026,3.0,0.0,,,,,0,,...,0,,,2,2,0,0,0,0,2
3975,2023_32024028_01,2023_32024028,3.0,0.0,,,,,7,5,...,7,0,30,2,3,0,0,2,2,2
